# 04 — Business & Policy Insights
From model to actionable policy recommendations for German SMEs

In [ ]:
import sys; sys.path.append('../src')
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, matplotlib.ticker as mticker
from data_loader import load_data, TARGET
from features import add_interactions, ALL_FEATURES
from model import load_model, split
from visualize import PALETTE, ORDER
df = load_data(); df = add_interactions(df)
X = df[[c for c in ALL_FEATURES if c in df.columns]]; y = df[TARGET]
X_train, X_test, y_train, y_test = split(X, y)

## 4.1 Management Quality Gap

In [ ]:
mgmt_idx = df.groupby(TARGET)['mgmt_quality_index'].mean().reindex(ORDER)
fig, ax = plt.subplots(figsize=(7,4))
mgmt_idx.plot.bar(ax=ax, color=[PALETTE[o] for o in ORDER], edgecolor='white')
ax.set_title('Average Management Quality Index by Productivity Class', fontsize=14)
ax.set_ylabel('Mean score (max = 4)'); ax.set_xlabel('')
plt.xticks(rotation=0); sns.despine(ax=ax)
plt.tight_layout(); plt.savefig('../reports/12_mgmt_gap.png',dpi=150); plt.show()

## 4.2 Green Practices & Productivity

In [ ]:
green_idx = df.groupby(TARGET)['green_score'].mean().reindex(ORDER)
fig, ax = plt.subplots(figsize=(7,4))
green_idx.plot.bar(ax=ax, color=[PALETTE[o] for o in ORDER], edgecolor='white')
ax.set_title('Average Green Score by Productivity Class', fontsize=14)
ax.set_ylabel('Mean score (max = 3)'); ax.set_xlabel('')
plt.xticks(rotation=0); sns.despine(ax=ax)
plt.tight_layout(); plt.savefig('../reports/13_green.png',dpi=150); plt.show()

## 4.3 What Holds Low-Productivity Firms Back?

In [ ]:
low_df = df[df[TARGET]=='Low']
obs_means = low_df[['k30_finance_obs','l30b_workforce_obs','j30a_tax_obs',
                    'j30f_corruption_obs','i30_crime_obs','d30a_transport_obs',
                    'l30a_labor_reg_obs','l30c_hiring_cost_obs','l30d_dismissal_obs']].mean()
obs_means.index = ['Finance','Skilled Workforce','Tax Rates','Corruption','Crime',
                   'Transport','Labour Regs','Hiring Cost','Dismissal Regs']
fig, ax = plt.subplots(figsize=(9,5))
obs_means.sort_values().plot.barh(ax=ax, color='#d73027', edgecolor='white')
ax.set_title('Mean Obstacle Severity — Low Productivity Firms (0=none, 4=very severe)', fontsize=13)
ax.set_xlabel('Mean severity score')
sns.despine(ax=ax); plt.tight_layout()
plt.savefig('../reports/14_obstacles_low.png',dpi=150); plt.show()

## 4.4 Market Scope vs Productivity

In [ ]:
scope_map = {1:'Local',2:'National',3:'International'}
df['market_label'] = df['e1_market_scope'].map(scope_map)
ct = df.groupby(['market_label',TARGET]).size().unstack(fill_value=0)
ct = ct.reindex(['Local','National','International'])[ORDER]
ax = ct.plot(kind='bar', color=[PALETTE[o] for o in ORDER], edgecolor='white', figsize=(9,5))
ax.set_title('Productivity Distribution by Market Scope', fontsize=14)
ax.set_xlabel(''); ax.legend(title='Productivity'); plt.xticks(rotation=0)
sns.despine(ax=ax); plt.tight_layout()
plt.savefig('../reports/15_market_scope.png',dpi=150); plt.show()

## 4.5 Policy Recommendation Summary

| Theme | Finding | Policy Recommendation |
|---|---|---|
| **Digital** | Website & R&D adoption 30-40pp higher in High-productivity firms | Subsidise digital onboarding; expand KfW R&D grants for SMEs |
| **Management** | KPI monitoring & performance targets strongly predict High class | Promote structured management training (Unternehmensberatung vouchers) |
| **Green** | CO2 monitoring & energy management rise with productivity | Link green subsidies to SME productivity metrics |
| **Finance** | Finance access is #1 obstacle for Low-productivity firms | Improve SME credit scoring; reduce collateral requirements |
| **Labour** | Training gap ~25pp; women training lower in Low-productivity firms | Mandate/incentivise Berufsschule-linked upskilling |
| **Market** | Internationally-oriented firms are 2x more likely to be High-productivity | Support export market entry for mid-sized SMEs |
| **Regulation** | High tax compliance burden correlates with Low productivity | Streamline e-filing; reduce hours spent on regulatory compliance |

*Source: World Bank Enterprise Survey Germany 2025 (`DEU_2025_WBES_v01_M`) — synthetic mirror.  
Replace `data/raw/DEU_2025_WBES_v01_M.csv` with real data for validated estimates.*